In [ ]:
%%configure
{"--enable-glue-datacatalog": "true"}

In [ ]:
%idle_timeout 60
%glue_version 5.1
%worker_type G.1X
%number_of_workers 2

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

In [ ]:
from pyspark.sql import functions as F
from functools import reduce

DB_SILVER = "tech_challenge_silver"
TABELAS = ["tb_sod_2023_2024", "tb_sod_2024_2025", "tb_sod_2025_2026"]

silver = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
                [spark.table(f"{DB_SILVER}.{t}") for t in TABELAS])

print("Total de linhas:", silver.count(), "| colunas:", len(silver.columns))
print([c for c in silver.columns if "ano" in c.lower() or "id" in c.lower() or "token" in c.lower()][:10])

In [ ]:
from pyspark.sql import functions as F

dfs = {t: spark.table(f"{DB_SILVER}.{t}") for t in TABELAS}

for t, d in dfs.items():
    r = d.agg(*[F.count(F.col(f"`{c}`")).alias(c) for c in d.columns]).collect()[0].asDict()
    print(f"## {t}: {d.count()} linhas, {len(d.columns)} colunas, {sum(v > 0 for v in r.values())} com dado")
    print("   colunas que começam com 'ano':", [c for c in d.columns if c.lower().startswith("ano")])
    if "p0_id_0" in d.columns:
        print("   ids distintos em p0_id_0:", d.select("p0_id_0").distinct().count())

c1, c2, c3 = [set(d.columns) for d in dfs.values()]
print("\nColunas em comum às 3:", len(c1 & c2 & c3))
print("2023 ∩ 2024:", len(c1 & c2), "| 2024 ∩ 2025:", len(c2 & c3))
print("Exemplos só na 2ª tabela:", sorted(c2 - c1)[:6])
print("Exemplos só na 3ª tabela:", sorted(c3 - c1 - c2)[:6])

In [ ]:
DB_SILVER = "tech_challenge_silver"

for t in ["tb_sod_2024_2025", "tb_sod_2025_2026"]:
    print(f"## {t}")
    print("\n".join(spark.table(f"{DB_SILVER}.{t}").columns))

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from functools import reduce
import re

DB_SILVER = "tech_challenge_silver"
TABELAS = ["tb_sod_2023_2024", "tb_sod_2024_2025", "tb_sod_2025_2026"]
ANOS = ["2023-2024", "2024-2025", "2025-2026"]

# 1 tabela por ano, identificada pelo valor de ano_pesquisa (não pelo nome da tabela)
brutas = {}
for t in TABELAS:
    d = spark.table(f"{DB_SILVER}.{t}")
    anos = [r[0] for r in d.select("ano_pesquisa").distinct().collect()]
    assert len(anos) == 1, f"{t} tem mais de um ano: {anos}"
    brutas[anos[0]] = d
assert sorted(brutas) == ANOS, f"anos encontrados: {sorted(brutas)}"


MAPA = {
 "id": ["p0_id_0", "col_0_a_token"],
 "idade": ["p1_a_idade_1", "col_1_a_idade"],
 "faixa_idade": ["p1_a_1_faixa_idade_2", "col_1_a_1_faixa_idade"],
 "genero": ["p1_b_genero_3", "col_1_b_genero"],
 "raca": ["p1_c_cor_raca_etnia_4", "col_1_c_cor_raca_etnia"],
 "pcd": ["p1_d_pcd_5", "col_1_d_pcd"],
 "uf": ["p1_i_1_uf_onde_mora_23", "col_1_i_1_uf_onde_mora"],
 "regiao": ["p1_i_2_regiao_onde_mora_24", "col_1_i_2_regiao_onde_mora"],
 "escolaridade": ["p1_l_nivel_de_ensino_27", "col_1_l_nivel_de_ensino"],
 "situacao_trabalho": ["p2_a_qual_sua_situacao_atual_de_trabalho_29", "col_2_a_situacao_de_trabalho"],
 "setor": ["p2_b_setor_30", "col_2_b_setor"],
 "porte_empresa": ["p2_c_numero_de_funcionarios_31", "col_2_c_numero_de_funcionarios"],
 "gestor": ["p2_d_gestor_32", "col_2_d_atua_como_gestor"],
 "cargo": ["p2_f_cargo_atual_34", "col_2_f_cargo_atual"],
 "nivel": ["p2_g_nivel_35", "col_2_g_nivel"],
 "faixa_salarial": ["p2_h_faixa_salarial_36", "col_2_h_faixa_salarial"],
 "tempo_exp_dados": ["p2_i_quanto_tempo_de_experiencia_na_area_de_dados_voce_tem_37", "col_2_i_tempo_de_experiencia_em_dados"],
 "satisfeito": ["p2_k_voce_esta_satisfeito_na_sua_empresa_atual_39", "col_2_k_satisfeito_atualmente"],
 "pretende_mudar": ["p2_n_voce_pretende_mudar_de_emprego_nos_proximos_6_meses_49", "col_2_n_planos_de_mudar_de_emprego_6m"],
 "modelo_trabalho_atual": ["p2_r_atualmente_qual_a_sua_forma_de_trabalho_62", "col_2_r_modelo_de_trabalho_atual", "col_2_q_modelo_de_trabalho_atual"],
 "modelo_trabalho_ideal": ["p2_s_qual_a_forma_de_trabalho_ideal_para_voce_63", "col_2_s_modelo_de_trabalho_ideal", "col_2_r_modelo_de_trabalho_ideal"],
 "prioridade_ia_empresa": ["p3_e_ai_generativa_e_uma_prioridade_em_sua_empresa_103", "col_3_e_ai_generativa_e_llm_e_uma_prioridade"],
 "usa_chatgpt_llm": ["p4_m_utiliza_chatgpt_ou_llms_no_trabalho_238", "col_4_m_usa_chatgpt_ou_copilot_no_trabalho", "col_4_j_usa_chatgpt_ou_copilot_no_trabalho"],
 "bons_resultados_llm": ["col_3_g_empresa_esta_conseguindo_ter_bons_resultados_com_llms"],  # só 2025-2026
}

# ---- Perguntas de múltipla escolha (várias colunas 0/1): regex por edição ----
# "nome" = casa os itens pelo nome | "posicao" = casa pela posição (1,2,3...) dentro da pergunta
def G(a, b, c, modo): return {"2023-2024": a, "2024-2025": b, "2025-2026": c, "modo": modo}
GRUPOS = {
 "linguagem":      G(r"^p4_d_\d+_", r"^col_4_d_\d+_", r"^col_4_c_\d+_", "nome"),
 "banco_de_dados": G(r"^p4_g_\d+_", r"^col_4_g_\d+_", r"^col_4_d_\d+_", "nome"),
 "cloud":          G(r"^p4_h_\d+_", r"^col_4_h_\d+_", r"^col_4_e_\d+_", "nome"),
 "ferramenta_bi":  G(r"^p4_j_\d+_", r"^col_4_j_\d+_", r"^col_4_g_\d+_", "nome"),
 "etl_engenharia": G(r"^p6_b_\d+_", r"^col_6_b_\d+_", r"^col_6_b_\d+_", "nome"),
 "ia_uso_individual": G(r"^p4_m_\d+_", r"^col_4_m_\d+_", r"^col_4_j_\d+_", "posicao"),
 "ia_uso_na_empresa": G(r"^p4_l_\d+_", r"^col_4_l_\d+_", r"^col_4_i_\d+_", "posicao"),
 "criterio_escolha_emprego": G(r"^p2_o_\d+_", r"^col_2_o_\d+_", r"^col_2_o_\d+_", "posicao"),
 "motivo_insatisfacao": G(None, r"^col_2_l_\d+_", r"^col_2_l_\d+_", "posicao"),  # 2023 tinha outras opções
 "desafio_gestor": G(r"^p3_d_\d+_", r"^col_3_d_\d+_", r"^col_3_d_\d+_", "posicao"),
 "barreira_ia_generativa": G(r"^p3_g_\d+_", r"^col_3_g_\d+_", r"^col_3_h_\d+_", "posicao"),
}

PREFIXO = r"^(?:p\d+|col_\d+)_[a-z]_\d+_"
REF = "2024-2025"   # edição usada como referência para os rótulos por posição

def pos(c):
    m = re.match(r"^(?:p\d+|col_\d+)_[a-z]_(\d+)_", c)
    return int(m.group(1)) if m else None

def limpo(c):
    return re.sub(r"_\d+$", "", re.sub(PREFIXO, "", c))

def colunas(df, ano, grupo):
    rx = GRUPOS[grupo][ano]
    return [c for c in df.columns if rx and re.match(rx, c)]

ROTULOS = {g: {pos(c): limpo(c) for c in colunas(brutas[REF], REF, g)}
           for g, s in GRUPOS.items() if s["modo"] == "posicao"}

def rotulo(grupo, c):
    if GRUPOS[grupo]["modo"] == "posicao":
        return ROTULOS[grupo].get(pos(c), limpo(c))
    return limpo(c)



In [ ]:
def to01(c):
    """1 = marcou, 0 = não marcou, NULL = não respondeu."""
    v = F.lower(F.trim(F.col(f"`{c}`").cast("string")))
    return (F.when(v.isNull() | (v == ""), F.lit(None).cast("int"))
             .when(v.isin("0", "0.0", "false", "nao", "não"), F.lit(0))
             .otherwise(F.lit(1)))

def montar_base(df, ano):
    sel, faltando = [], []
    for nome, candidatos in MAPA.items():
        achado = next((c for c in candidatos if c in df.columns), None)
        if achado:
            sel.append(F.col(f"`{achado}`").cast("string").alias(nome))
        else:
            sel.append(F.lit(None).cast("string").alias(nome))
            faltando.append(nome)

    # Uso individual de IA: posição 1 = "não uso"; 2 a 5 = formas de uso
    cols_ia = colunas(df, ano, "ia_uso_individual")
    assert cols_ia, f"{ano}: colunas de uso de IA não encontradas"
    cols_uso = [c for c in cols_ia if pos(c) in (2, 3, 4, 5)]
    resp = reduce(lambda a, b: a | b, [F.col(f"`{c}`").isNotNull() for c in cols_ia])
    uso = F.greatest(*[to01(c) for c in cols_uso])
    sel += [resp.alias("respondeu_ia"),
            F.when(resp, F.coalesce(uso, F.lit(0))).alias("usa_ia")]

    p = df.select(*sel).withColumn("ano", F.lit(ano))

    g = F.lower(F.trim(F.coalesce(F.col("genero"), F.lit(""))))

    def modelo(c):
        m = F.lower(F.coalesce(F.col(c), F.lit("")))
        return (F.when(m.contains("100% remoto"), "Remoto")
                 .when(m.contains("flex"), "Híbrido flexível")
                 .when(m.contains("dias fixos"), "Híbrido dias fixos")
                 .when(m.contains("100% presencial"), "Presencial"))

    pri = F.lower(F.coalesce(F.col("prioridade_ia_empresa"), F.lit("")))

    # faixa salarial (texto) -> ponto médio numérico (aproximação)
    t = F.regexp_replace(F.lower(F.coalesce(F.col("faixa_salarial"), F.lit(""))), r"\.", "")
    a = F.regexp_extract(t, r"(\d+)", 1)
    b = F.regexp_extract(t, r"(\d+)\D+(\d+)", 2)
    n1 = F.when(F.length(a) > 0, a.cast("double"))
    n2 = F.when(F.length(b) > 0, b.cast("double"))
    sal = (F.when(n2.isNotNull(), (n1 + n2) / 2)
            .when(t.contains("menos"), n1 / 2)
            .when(n1.isNotNull(), n1))

    p = (p.withColumn("genero_grupo",
                      F.when(g.contains("fem") | g.contains("mulher"), "Feminino")
                       .when(g.contains("masc") | g.contains("homem"), "Masculino")
                       .when(g == "outro", "Outro")
                       .otherwise("Não informado"))
          .withColumn("modelo_trabalho", modelo("modelo_trabalho_atual"))
          .withColumn("modelo_ideal", modelo("modelo_trabalho_ideal"))
          .withColumn("salario_ponto_medio", sal)
          .withColumn("prioridade_ia_empresa",
                      F.when(pri.contains("nossa principal prioridade"), "Principal prioridade")
                       .when(pri.contains("entre nossas principais"), "Entre as principais")
                       .when(pri.contains("mais ou menos"), "Uma entre várias iniciativas")
                       .when(pri.contains("não é uma iniciativa"), "Não é prioridade")
                       .when(pri.contains("não sei opinar"), "Não sabe opinar")))
    return p, faltando

In [ ]:
print("ok", list(brutas.keys()))

In [ ]:
for nome in ["brutas", "MAPA", "GRUPOS", "ROTULOS", "montar_base", "to01"]:
    print(nome, "->", "definido" if nome in globals() else "FALTA")

In [ ]:
ESSENCIAIS = ["id", "genero", "nivel", "cargo", "faixa_salarial", "regiao"]
partes = []

for ano in ANOS:
    b, faltando = montar_base(brutas[ano], ano)
    n = b.count()
    ids = b.select("id").distinct().count()
    nn = b.agg(*[F.count(c).alias(c) for c in ESSENCIAIS + ["salario_ponto_medio", "usa_ia"]]).collect()[0].asDict()
    print(f"{ano}: {n} linhas | {ids} ids distintos | campos sem coluna no MAPA: {faltando}")
    print("   preenchidos:", nn)
    vazios = [c for c in ESSENCIAIS if nn[c] == 0]
    assert not vazios, f"{ano}: colunas essenciais vazias {vazios} -> ajustar o MAPA"
    partes.append(b)

base = reduce(lambda a, b: a.unionByName(b), partes).cache()
print("TOTAL:", base.count())

In [ ]:
for c in ["genero_grupo", "nivel", "regiao", "modelo_trabalho", "prioridade_ia_empresa", "usa_chatgpt_llm", "bons_resultados_llm"]:
    print("##", c)
    base.groupBy("ano", c).count().orderBy("ano", F.desc("count")).show(12, truncate=50)

print("## faixas salariais que não viraram número:")
(base.filter(F.col("faixa_salarial").isNotNull() & F.col("salario_ponto_medio").isNull())
     .groupBy("ano", "faixa_salarial").count().show(15, truncate=60))

base.groupBy("ano").agg(F.count("*").alias("n"),
                        F.round(F.avg("salario_ponto_medio"), 0).alias("salario_medio"),
                        F.round(100 * F.avg("usa_ia"), 1).alias("pct_usa_ia")).orderBy("ano").show()

In [ ]:
b = base.filter("ano = '2025-2026'")
for c in ["regiao", "modelo_trabalho", "prioridade_ia_empresa"]:
    print("##", c)
    b.groupBy(c).count().orderBy(F.desc("count")).show(10, truncate=40)

In [ ]:
def distribuicao(df, col, nome):
    w = Window.partitionBy("ano")
    return (df.filter(F.col(col).isNotNull())
              .groupBy("ano", F.col(col).alias("categoria")).agg(F.count("*").alias("qtd"))
              .withColumn("dimensao", F.lit(nome))
              .withColumn("pct", F.round(100 * F.col("qtd") / F.sum("qtd").over(w), 2))
              .select("ano", "dimensao", "categoria", "qtd", "pct"))

def segmentar(df, dim):
    seg = F.lit("total") if dim == "total" else F.coalesce(F.col(dim), F.lit("Não informado"))
    return df.withColumn("segmento", seg).withColumn("dimensao", F.lit(dim))

def unpivot(grupo):
    """Múltipla escolha (colunas 0/1) -> linhas: ano, categoria, item, qtd, base, pct."""
    saidas = []
    for ano in ANOS:
        df = brutas[ano]
        cols = colunas(df, ano, grupo)
        if not cols:
            continue
        resp = reduce(lambda a, b: a | b, [F.col(f"`{c}`").isNotNull() for c in cols])
        d = df.filter(resp)
        n = d.count()
        if n == 0:
            continue
        arr = F.array(*[F.struct(F.lit(rotulo(grupo, c)).alias("item"), to01(c).alias("v")) for c in cols])
        q = (d.select(F.explode(arr).alias("x"))
               .groupBy(F.col("x.item").alias("item"))
               .agg(F.sum(F.coalesce(F.col("x.v"), F.lit(0))).cast("long").alias("qtd"))
               .withColumn("ano", F.lit(ano)).withColumn("categoria", F.lit(grupo))
               .withColumn("base_respondentes", F.lit(n))
               .withColumn("pct", F.round(100 * F.col("qtd") / F.col("base_respondentes"), 2)))
        saidas.append(q.select("ano", "categoria", "item", "qtd", "base_respondentes", "pct"))
    return reduce(lambda a, b: a.unionByName(b), saidas)

def uniao(lst):
    return reduce(lambda a, b: a.unionByName(b), lst)

In [ ]:
# 1. Estrutura do mercado
gold_perfil_mercado = uniao([distribuicao(base, c, c) for c in
    ["regiao", "uf", "setor", "porte_empresa", "cargo", "nivel", "escolaridade",
     "faixa_idade", "situacao_trabalho", "modelo_trabalho", "modelo_ideal", "tempo_exp_dados"]])

# 2. Perfis mais valorizados
gold_salario_por_perfil = (base.filter(F.col("salario_ponto_medio").isNotNull())
    .groupBy("ano", "cargo", "nivel")
    .agg(F.count("*").alias("n"),
         F.round(F.avg("salario_ponto_medio"), 0).alias("salario_medio"),
         F.round(F.percentile_approx("salario_ponto_medio", 0.5), 0).alias("salario_mediano")))

# 3. Diversidade de gênero
def diversidade(dim):
    d = segmentar(base, dim)
    w = Window.partitionBy("ano", "dimensao", "segmento")
    return (d.groupBy("ano", "dimensao", "segmento", "genero_grupo")
              .agg(F.count("*").alias("qtd"),
                   F.round(F.avg("salario_ponto_medio"), 0).alias("salario_medio"))
              .withColumn("pct_no_segmento", F.round(100 * F.col("qtd") / F.sum("qtd").over(w), 2)))

gold_diversidade_genero = uniao([diversidade(d) for d in ["total", "cargo", "nivel", "regiao"]])

# 4. Tecnologias e uso de IA
gold_tecnologias = uniao([unpivot(g) for g in
    ["linguagem", "banco_de_dados", "cloud", "ferramenta_bi", "etl_engenharia",
     "ia_uso_individual", "ia_uso_na_empresa"]])

# 5. Adoção de IA (associação, não causa) e visão das empresas
def adocao_ia(dim):
    d = segmentar(base.filter(F.col("respondeu_ia")), dim)
    return (d.groupBy("ano", "dimensao", "segmento")
              .agg(F.count("*").alias("respondentes"),
                   F.sum("usa_ia").alias("usam_ia"),
                   F.round(F.avg(F.when(F.col("usa_ia") == 1, F.col("salario_ponto_medio"))), 0).alias("salario_medio_usuarios"),
                   F.round(F.avg(F.when(F.col("usa_ia") == 0, F.col("salario_ponto_medio"))), 0).alias("salario_medio_nao_usuarios"))
              .withColumn("pct_adocao", F.round(100 * F.col("usam_ia") / F.col("respondentes"), 2)))

gold_adocao_ia = uniao([adocao_ia(d) for d in ["total", "cargo", "nivel", "regiao", "modelo_trabalho"]])

gold_ia_empresa = uniao([distribuicao(base, "prioridade_ia_empresa", "prioridade_ia_generativa"),
                         distribuicao(base, "bons_resultados_llm", "bons_resultados_com_llms")])

# 6. Regiões, senioridade e modelo de trabalho
gold_comparativo_segmentos = (base.groupBy("ano", "regiao", "nivel", "modelo_trabalho")
    .agg(F.count("*").alias("n"),
         F.round(F.avg("salario_ponto_medio"), 0).alias("salario_medio"),
         F.round(F.percentile_approx("salario_ponto_medio", 0.5), 0).alias("salario_mediano"),
         F.round(100 * F.avg("usa_ia"), 2).alias("pct_usa_ia")))

# 7. Desafios e oportunidades
gold_desafios_oportunidades = uniao([unpivot(g) for g in
    ["criterio_escolha_emprego", "motivo_insatisfacao", "desafio_gestor", "barreira_ia_generativa"]])

gold = {
    "gold_perfil_mercado": gold_perfil_mercado,
    "gold_salario_por_perfil": gold_salario_por_perfil,
    "gold_diversidade_genero": gold_diversidade_genero,
    "gold_tecnologias": gold_tecnologias,
    "gold_adocao_ia": gold_adocao_ia,
    "gold_ia_empresa": gold_ia_empresa,
    "gold_comparativo_segmentos": gold_comparativo_segmentos,
    "gold_desafios_oportunidades": gold_desafios_oportunidades,
}
print("Gold montada (ainda não gravada):", list(gold))

In [ ]:
loc = [r for r in spark.sql(f"DESCRIBE TABLE EXTENDED {DB_SILVER}.{TABELAS[0]}").collect() if r[0] == "Location"][0][1]
BUCKET = loc.split("/")[2]
GOLD_BASE = f"s3://{BUCKET}/gold"
DB_GOLD = "tech_challenge_gold"
print("Silver em:", loc, "\nGold em:", GOLD_BASE)

spark.sql(f"CREATE DATABASE IF NOT EXISTS {DB_GOLD} LOCATION '{GOLD_BASE}/'")

for nome, df in gold.items():
    n = df.count()
    assert n > 0, f"{nome} está vazia"
    (df.write.mode("overwrite").format("parquet")
       .option("path", f"{GOLD_BASE}/{nome}/").saveAsTable(f"{DB_GOLD}.{nome}"))
    df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{GOLD_BASE}/csv/{nome}/")
    print(f"{nome}: {n} linhas")

In [ ]:
for t in gold:
    d = spark.table(f"{DB_GOLD}.{t}")
    print(t, d.groupBy("ano").count().orderBy("ano").collect())

spark.table(f"{DB_GOLD}.gold_tecnologias") \
    .filter("categoria = 'linguagem'") \
    .orderBy(F.desc("pct")).select("ano", "item", "qtd", "pct").show(12, truncate=30)

In [ ]:
import boto3
from collections import Counter

s3 = boto3.client("s3")
BUCKET = "lab-teste-tech-challenge-6890"
cont = Counter()

for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix="gold/"):
    for o in page.get("Contents", []):
        partes = o["Key"].split("/")
        pasta = "/".join(partes[:3]) if partes[1] == "csv" else "/".join(partes[:2])
        ext = partes[-1].rsplit(".", 1)[-1] if "." in partes[-1] else "(sem extensão)"
        cont[(pasta, ext)] += 1

for (pasta, ext), n in sorted(cont.items()):
    print(f"{pasta:50s} .{ext:15s} {n}")

In [ ]:
for r in spark.sql("DESCRIBE TABLE EXTENDED tech_challenge_gold.gold_perfil_mercado").collect():
    if r[0] in ("Provider", "Location", "InputFormat", "Serde Library"):
        print(r[0], "->", r[1])

In [ ]:
for t in ["gold_perfil_mercado", "gold_salario_por_perfil", "gold_diversidade_genero",
          "gold_tecnologias", "gold_adocao_ia", "gold_ia_empresa",
          "gold_comparativo_segmentos", "gold_desafios_oportunidades"]:
    spark.sql(f"ALTER TABLE tech_challenge_gold.{t} SET TBLPROPERTIES ('classification'='parquet')")
    print("ok", t)

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.gold_ia_empresa")
spark.sql(f"SHOW TABLES IN {DB_GOLD}").show(truncate=False)

In [ ]:
adocao = uniao([adocao_ia(d) for d in ["total", "cargo", "nivel", "regiao", "modelo_trabalho"]])

empresa = (uniao([distribuicao(base, "prioridade_ia_empresa", "prioridade_ia_generativa"),
                  distribuicao(base, "bons_resultados_llm", "bons_resultados_com_llms")])
           .withColumnRenamed("categoria", "segmento")
           .withColumnRenamed("qtd", "respondentes")
           .withColumnRenamed("pct", "pct_no_total"))

df = adocao.unionByName(empresa, allowMissingColumns=True)
gold["gold_adocao_ia"] = df
gold.pop("gold_ia_empresa", None)

print(df.groupBy("dimensao").count().orderBy("dimensao").collect())

spark.sql(f"DROP TABLE IF EXISTS {DB_GOLD}.gold_adocao_ia")
(df.coalesce(1).write.mode("overwrite").format("parquet")
   .option("path", f"{GOLD_BASE}/gold_adocao_ia/").saveAsTable(f"{DB_GOLD}.gold_adocao_ia"))
df.coalesce(1).write.mode("overwrite").option("header", True).csv(f"{GOLD_BASE}/csv/gold_adocao_ia/")
print("gold_adocao_ia:", df.count(), "linhas")


In [ ]:
spark.sql(f"SHOW TABLES IN {DB_GOLD}").show(truncate=False)

In [ ]:
spark.sql("ALTER TABLE tech_challenge_gold.gold_adocao_ia SET TBLPROPERTIES ('classification'='parquet')")
print("ok")